In the previous notebook, we saw how embedding transforms text into a numerical vector. It enables semantic search, allowing queries to be matched based on meaning and context rather than exact keywords. Combined with vector indexing, embeddings enable fast similarity searches to retrieve the most relevant context for an LLM prompt. The notebook is a PoC to retrieve information from graph Retrieval-Augmented Generation (RAG). It combines Retrieval-Augmented Generation with a graph model that can capture relationships between entities.

The vector lenght depends of the encoder model used. In the PoC we used ```bge-m3``` as it is multilingual and provides good results according to the current state of the art.

In [1]:
import json
from neo4j import GraphDatabase
import ollama
from sentence_transformers import SentenceTransformer
import os
import hashlib
import sys
sys.path.insert(1, "src/graph/")
from graph_builder import (
    get_node_id,
    extract_graph,
    compute_chunk_embeddings,
    merge_graphs,
    validate_graph,
    add_speaker_entities,
    build_neo4j_graph,
    feed_global_report,
    save_graph,
    load_to_neo4j,
    create_constraints
)
sys.path.insert(1, "src/preprocessing/")
from speeches import (
    load_speeches,
    split_into_chunks,
    load_prompt_template
)

# Open neo4j and load credential

The first step consit to start neo4j driver. Indeed we use it to store our data and index our embedded vector. The engine optimize reseach index. It allows to retrieve best result for cosine similarity faster. be sure to create `credential.json` in the same folder as the notebook. 

```
{
    "USER" : XXX,
    "PASSWORD" : YYY,
    "URI" : "bolt://127.0.0.1:7687"
}
```

With:
* `XXX` your USER ID in Neo4J 
* ``YYY` your password

NB: Because our database is local host the url is `"URI" : "bolt://127.0.0.1:7687"` 


Before to use the notebook, start Neo4J Desktop without that the driver will not be able to connect to neo4j.

In [2]:
# Check the presence of credential.json 
if os.path.exists("credential.json"):
    with open("credential.json") as json_file:
        credential = json.load(json_file)
    print("Check connectivity with neo4j")
    print("need to lunch Neo4j desktop and DB before")
    driver = GraphDatabase.driver(
        credential["URI"],
        auth=(
            credential["USER"],
            credential["PASSWORD"]
        )
    )
    driver.verify_connectivity()
    print("Load cypher script for constraint")
    create_constraints(
        driver,
        os.path.join("src", "graph", "graph_constraint.txt")
    )
    print("Neo4j is connected !")
else:
    print("JSON not found for neo4j parameter")
    exit()

Check connectivity with neo4j
need to lunch Neo4j desktop and DB before
Load cypher script for constraint
Neo4j constraints created.
Neo4j is connected !


In [3]:
def get_schema_from_driver(driver):
    """_summary_

    Args:
        driver (_type_): _description_

    Returns:
        _type_: _description_
    """
    with driver.session() as session:
        result = session.run("CALL db.schema.visualization()")
        record = result.single()
        nodes = [f"(:{node['name']})" for node in record["nodes"]]
        relationships = [
            f"(:{rel.start_node['name']})-[:{rel.type}]->(:{rel.end_node['name']})"
            for rel in record["relationships"]
        ]
        return f"Nodes:\n" + "\n".join(nodes) + "\n\nRelationships:\n" + "\n".join(relationships)

schema_text = get_schema_from_driver(driver)
print(schema_text)

Nodes:
(:Theme)
(:Event)
(:Organization)
(:Law)
(:Country)
(:Speech)
(:Person)
(:City)
(:Chunk)

Relationships:
(:Speech)-[:HAS_CHUNK]->(:Chunk)
(:Person)-[:MENTIONS]->(:City)
(:Theme)-[:MENTIONS]->(:Law)
(:Organization)-[:MENTIONS]->(:Law)
(:Person)-[:MENTIONS]->(:Theme)
(:Chunk)-[:MENTIONS]->(:Organization)
(:Organization)-[:MENTIONS]->(:Country)
(:Law)-[:MENTIONS]->(:Law)
(:Chunk)-[:MENTIONS]->(:Person)
(:City)-[:MENTIONS]->(:Event)
(:Person)-[:MENTIONS]->(:Person)
(:Law)-[:MENTIONS]->(:Organization)
(:Country)-[:MENTIONS]->(:Theme)
(:Chunk)-[:MENTIONS]->(:City)
(:Person)-[:MENTIONS]->(:Organization)
(:Organization)-[:MENTIONS]->(:Person)
(:Theme)-[:MENTIONS]->(:Organization)
(:Country)-[:MENTIONS]->(:Organization)
(:Theme)-[:MENTIONS]->(:Person)
(:Event)-[:MENTIONS]->(:Event)
(:Country)-[:MENTIONS]->(:City)
(:Country)-[:MENTIONS]->(:Person)
(:Law)-[:MENTIONS]->(:Person)
(:Event)-[:MENTIONS]->(:Person)
(:City)-[:MENTIONS]->(:Law)
(:City)-[:MENTIONS]->(:Person)
(:Chunk)-[:MENTIONS]->

# Do we need an agent ?

An agent is usefull when we want trigger different pipeline according to user context. 
We do not need an agent. Indeed we limited the PoC to the simplest usecase.
It is a chatbot that retrieves results with cypher query generated by a LLM.

# Use as classic RAG

neo4j indexes the data in an efficient way. We can process semantic search to find speeches that match to our sentence.
remember semantic search use cosine similarity. The first step concist to embedded our text with ```bge-m3```. 

In [4]:
def text2vector(text : str, model : SentenceTransformer = None) -> list:
    """convert text to vector

    Args:
        text (str): text
        model (SentenceTransformer, optional): encoder use to transform texxt to vector. Defaults to SentenceTransformer("BAAI/bge-m3").

    Returns:
        list[np.ndarray]: embedded vector
    """
    if model is None:
        model = SentenceTransformer("BAAI/bge-m3")
    embedding = model.encode(
        text,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    # Convert into list
    return embedding.tolist()

In [5]:
embedding_model = SentenceTransformer("BAAI/bge-m3")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [6]:
text = "Le président Emmanuel Macron parle émeutes en France"
query_embeddings = text2vector(text, embedding_model)

In the graph database each chunk have the attribute `embedding`. Before to compare our `query_embeddings`, we create vector index.The indexation step will sort the embedded vector in a way more effecient for similarity comparison. The following query will create the index.

In [7]:
def create_vector_index(driver):
    cypher_query = """
    CREATE VECTOR INDEX chunk_embeddings IF NOT EXISTS
    FOR (c:Chunk) ON (c.embedding)
    OPTIONS {
      indexConfig: {
        `vector.dimensions`: 1024,
        `vector.similarity_function`: 'cosine'
      }
    }
    """
    with driver.session() as session:
        session.run(cypher_query)
    print("Index vectoriel 'chunk_embeddings' check/create.")

In [8]:
# call the driver
create_vector_index(driver)

Index vectoriel 'chunk_embeddings' check/create.


After the index creation we can collect the 5 best chunk with the following query

In [9]:
cypher_query = """
CALL db.index.vector.queryNodes('chunk_embeddings', 20, $query_embeddings)
YIELD node AS chunk, score
MATCH (speech:Speech)-[:HAS_CHUNK]->(chunk)
RETURN speech.id AS speech_id, 
       speech.title AS title, 
       speech.date AS date, 
       max(score) AS similarity_score,
       collect(chunk.text)[0] AS best_excerpt
ORDER BY similarity_score DESC
LIMIT 5
"""

In [10]:
# execute the query into session
with driver.session() as session:
    result = session.run(cypher_query, query_embeddings=query_embeddings)
    
    print("TOP 5 Speeches")
    for record in result:
        print(f"Score: {record['similarity_score']:.4f} | Date: {record['date']}")
        print(f"Titre: {record['title']}")
        print(f"Extrait: {record['best_excerpt']}...\n\n")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('chunk_embeddings', 20, $query_embeddings)\nYIELD node AS chunk, score\nMATCH (speech:Speech)-[:HAS_CHUNK]->(chunk)\nRETURN speech.id AS speech_id, \n       speech.title AS title, \n       speech.date AS date, \n       max(score) AS similarity_score,\n       collect(chunk.text)[0] AS best_excerpt\nORDER BY s

TOP 5 Speeches
Score: 0.8280 | Date: 2018-12-10
Titre: Allocution télévisée de M. Emmanuel Macron, Président de la République, sur les réponses du gouvernement à l'urgence économique et sociale révélée par la contestation des "Gilets jaunes", Paris le 10 décembre 2018.
Extrait: Françaises, Français, nous voilà ensemble au rendez-vous de notre pays et de notre avenir. Les événements de ces dernières semaines dans l'Hexagone et outremer ont profondément troublé la Nation. Ils ont mêlé des revendications légitimes et un enchaînement de violences inadmissibles et je veux vous le dire d'emblée : ces violences ne bénéficieront d'aucune indulgence.
Nous avons tous vu le jeu des opportunistes qui ont essayé de profiter des colères sincères pour les dévoyer. Nous avons tous vu les irresponsables politiques dont le seul projet était de bousculer la République, cherchant le désordre et l'anarchie. Aucune colère ne justifie qu'on s'attaque à un policier, à un gendarme, qu'on dégrade un commerce ou

# Call LLM

Write cypher query can be defficult for begining user. We can use LLM model to translate natural language into a cypher query. We declare the database schema at the begining and gave example into the prompt.

The result will depend on the model. This time we use `qwen2.5-coder:7b`.The dataset for the training was based on code script. 

NB: Neo4j made a fine-tuned model to transform natural language to cypher query. For the tutorial we stay with qwen model to be the more user friendly.

NB²: The prompt was designed after many attempt and a final refinement was made with Ggoogle Gemini. 

In [22]:
NEO4J_SCHEMA_PROMPT = """
Tu es un expert Cypher pour Neo4j. Ta tâche est de traduire une question en une requête Cypher exacte.

### SCHÉMA DE LA BASE DE DONNÉES

NŒUDS :
- Speech {id, title, date, filename}
- Chunk {id, text, embedding}
- Person {name}
- Organization {name}
- Country {name}
- City {name}
- Law {name}
- Event {name}
- Theme {name}

RELATIONS AUTORISÉES (Strictement respecter le sens x -> y) :
- (Speech)-[:DELIVERED_BY]->(Person)
- (Speech)-[:HAS_CHUNK]->(Chunk)
- (Chunk)-[:MENTIONS]->(Person|Organization|Country|City|Law|Event|Theme)
- (Person|Organization|Country)-[:PROPOSED]->(Law)
- (Person|Organization|Country)-[:SUPPORTS|OPPOSES]->(Law|Theme|Event)

RÈGLES STRICTES :
1. N'invente AUCUNE relation. Utilise uniquement les relations ci-dessus.
2. Pour lier un Speech à un sujet/loi/pays, passe TOUJOURS par (s:Speech)-[:HAS_CHUNK]->(c:Chunk)-[:MENTIONS]->(entité).
3. Utilise toLower(n.name) pour les filtres de texte.
4. Réponds UNIQUEMENT avec la requête Cypher, sans explication, sans backticks ```.

RÈGLES DE SÉLECTION D'INDEX :
1. Si la question contient des termes comme "sémantique", "recherche vectorielle", "parle de", "sujet", "thème" ou une notion abstraite :
   -> Utilise IMPÉRATIVEMENT l'index vectoriel :
   CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score

2. N'utilise les filtres d'entités (:MENTIONS->(:Country)) QUE si l'utilisateur demande explicitement une relation structurée avec un pays/entité nommée (ex: "pays mentionné").

RÈGLE D'EXCLUSION :
Lorsqu'une requête utilise 'db.index.vector.queryNodes', NE RAJOUTE PAS de clauses 'MATCH (c)-[:MENTIONS]->(...)' sauf si la question demande explicitement une relation d'entité spécifique.

### EXEMPLES :

Question : Quels sont les discours d'Emmanuel Macron ?
Cypher : MATCH (s:Speech)-[:DELIVERED_BY]->(p:Person) WHERE toLower(p.name) CONTAINS 'macron' RETURN s.title, s.date

Question : Quels sont les pays mentionnés dans les discours ?
Cypher : MATCH (s:Speech)-[:HAS_CHUNK]->(c:Chunk)-[:MENTIONS]->(co:Country) RETURN DISTINCT co.name

Question : Quels discours parlent des lois proposées par la France ?
Cypher : MATCH (co:Country)-[:PROPOSED]->(l:Law)<-[:MENTIONS]-(c:Chunk)<-[:HAS_CHUNK]-(s:Speech) WHERE toLower(co.name) = 'france' RETURN DISTINCT s.title, s.date, s.filename

Question : Donne moi le top 5 des discours où le président Emmanuel Macron parle des émeutes en France
Cypher : CALL db.index.vector.queryNodes('chunk_embeddings', 50, $query_embedding) YIELD node AS c, score MATCH (s:Speech)-[:HAS_CHUNK]->(c) MATCH (s)-[:DELIVERED_BY]->(p:Person) WHERE toLower(p.name) CONTAINS 'macron' RETURN s.title, s.date, s.filename, max(score) AS max_score ORDER BY max_score DESC LIMIT 5

Question : Dans combien de pays différents de la France le président français a fait des discours ?
Cypher : MATCH (s:Speech)-[:DELIVERED_BY]->(p:Person) WHERE toLower(p.name) CONTAINS 'macron' MATCH (s)-[:HAS_CHUNK]->(c:Chunk)-[:MENTIONS]->(co:Country) WHERE toLower(co.name) <> 'france' RETURN count(DISTINCT co) AS country_count

Question : Combien de discours parle de la guerre en Ukraine ? Je veux une confidence de 0.6 au moins
Cypher : CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score WHERE score >= 0.6 MATCH (s:Speech)-[:HAS_CHUNK]->(c) RETURN count(DISTINCT s) AS total_speeches

Question : Combien de discours parlent de la guerre en Ukraine ?
Cypher : CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score MATCH (s:Speech)-[:HAS_CHUNK]->(c) RETURN count(DISTINCT s) AS total_speeches

EXEMPLE POUR RECHERCHE D'ACCOMPAGNANTS / PERSONNES MENTIONNÉES :
Question : Avec qui était le président lorsqu'il parlait de la Corse ?
Cypher : CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score MATCH (s:Speech)-[:HAS_CHUNK]->(c) MATCH (c)-[:MENTIONS]->(p:Person) WHERE NOT toLower(p.name) CONTAINS 'macron' RETURN DISTINCT p.name AS acompanhate_name
"""

Now we can create a function to concatenate the prompt with the user's question and then generate the cypher query with a LLM

In [12]:
def generate_cypher_query(user_question: str, NEO4J_SCHEMA_PROMPT: str, model: str = "qwen2.5-coder:7b") -> str:
    """_summary_

    Args:
        user_question (str): _description_
        model (_type_, optional): _description_. Defaults to "qwen2.5-coder:7b".

    Returns:
        str: _description_
    """
    
    messages = [
        {"role": "system", "content": NEO4J_SCHEMA_PROMPT},
        {"role": "user", "content": f"Question : {user_question}"}
    ]
    
    response = ollama.chat(
        model=model,
        messages=messages,
        options={
            "temperature": 0  # Température 0 pour garantir la précision de la syntaxe
        }
    )
    
    cypher_query = response["message"]["content"].strip()
    
    # Nettoyage si le modèle ajoute malgré tout des balises markdown
    if cypher_query.startswith("```cypher"):
        cypher_query = cypher_query.replace("```cypher", "").replace("```", "").strip()
    elif cypher_query.startswith("```"):
        cypher_query = cypher_query.replace("```", "").strip()
        
    return cypher_query

In [13]:
question = "Quels discours parlent des lois proposées par la France ?"
cypher = generate_cypher_query(question, NEO4J_SCHEMA_PROMPT)

print("Cypher query")
print(cypher)

Cypher query
MATCH (co:Country)-[:PROPOSED]->(l:Law)<-[:MENTIONS]-(c:Chunk)<-[:HAS_CHUNK]-(s:Speech) WHERE toLower(co.name) = 'france' RETURN DISTINCT s.title, s.date, s.filename


Now we have a function that generates the cypher prompt (`generate_cypher_query`), we can use it into neo4j. We have just take in count to replace the argument `$query_embedding` in the query by the embedded vector. By this way we can process semantic search

In [14]:
def ask_graph(question: str, NEO4J_SCHEMA_PROMPT: str, driver, model="qwen2.5-coder:7b"):
    """_summary_

    Args:
        question (str): _description_
        NEO4J_SCHEMA_PROMPT (str): _description_
        driver (_type_): _description_
        model (str, optional): _description_. Defaults to "qwen2.5-coder:7b".

    Returns:
        _type_: _description_
    """
    # Create cypher query
    cypher_query = generate_cypher_query(question, NEO4J_SCHEMA_PROMPT, model=model)
    print(f"\nCypher query\n{cypher_query}\n")
    
    # replace $query_embedding 
    params = {}
    if "$query_embedding" in cypher_query:
        # create the embedded vector
        params["query_embedding"] = text2vector(question)
    
    # Use the query
    with driver.session() as session:
        result = session.run(cypher_query, parameters=params)
        return [record.data() for record in result]


In [17]:
question = "Combien de discours parle de la guerre en Ukraine ? Je veux une confidence de 0.6 au moins"
ask_graph(question, NEO4J_SCHEMA_PROMPT, driver)


Cypher query
CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score WHERE score >= 0.6 MATCH (s:Speech)-[:HAS_CHUNK]->(c) RETURN count(DISTINCT s) AS total_speeches



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score WHERE score >= 0.6 MATCH (s:Speech)-[:HAS_CHUNK]->(c) RETURN count(DISTINCT s) AS total_speeches"


[{'total_speeches': 46}]

In [18]:
question = "Avec une recherche sémantique trouve combien de discours parlent de la Nouvelle Calédonie"
ask_graph(question, NEO4J_SCHEMA_PROMPT, driver)


Cypher query
CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score MATCH (s:Speech)-[:HAS_CHUNK]->(c) RETURN count(DISTINCT s) AS total_speeches



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score MATCH (s:Speech)-[:HAS_CHUNK]->(c) RETURN count(DISTINCT s) AS total_speeches"


[{'total_speeches': 35}]

In [19]:
question ="donne moi le top 3 des discours où le président Emmanuel Macron parle de la retraite en France"
ask_graph(question, NEO4J_SCHEMA_PROMPT, driver)


Cypher query
CALL db.index.vector.queryNodes('chunk_embeddings', 50, $query_embedding) YIELD node AS c, score MATCH (s:Speech)-[:HAS_CHUNK]->(c) MATCH (s)-[:DELIVERED_BY]->(p:Person) WHERE toLower(p.name) CONTAINS 'macron' RETURN s.title, s.date, s.filename, max(score) AS max_score ORDER BY max_score DESC LIMIT 3



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('chunk_embeddings', 50, $query_embedding) YIELD node AS c, score MATCH (s:Speech)-[:HAS_CHUNK]->(c) MATCH (s)-[:DELIVERED_BY]->(p:Person) WHERE toLower(p.name) CONTAINS 'macron' RETURN s.title, s.date, s.filename, max(score) AS max_score ORDER BY max_score DESC LIMIT 3"


[{'s.title': 'Déclaration et réponses à des questions de M. Emmanuel Macron, Président de la République, sur la réforme des retraites, à Rodez le 3 octobre 2019.',
  's.date': '2019-10-03',
  's.filename': '2019-10-03_d-claration-et-r-ponses-des-questions-de-m-emmanuel-macron-p.txt',
  'max_score': 0.7941879630088806},
 {'s.title': "Interview de M. Emmanuel Macron, président de la république, à TF1 et France 2 le 14 juillet 2020, sur l'épidémie de Covid-19, la crise économique et les défis et priorités de la suite de son quinquennat.",
  's.date': '2020-07-14',
  's.filename': '2020-07-14_interview-de-m-emmanuel-macron-pr-sident-de-la-r-publique-tf.txt',
  'max_score': 0.7896053791046143},
 {'s.title': "Interview de M. Emmanuel Macron, Président de la République, à France 2 le 26 août 2019, sur le Sommet du G7, le maintien de l'ordre en France et la réforme des retraites.",
  's.date': '2019-08-26',
  's.filename': '2019-08-26_interview-de-m-emmanuel-macron-pr-sident-de-la-r-publique-f

In [24]:
question = "Avec qui était le préseident lorsqu'il parlait de la nouvelle calédonie ?"
ask_graph(question, NEO4J_SCHEMA_PROMPT, driver)


Cypher query
CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score MATCH (s:Speech)-[:HAS_CHUNK]->(c) MATCH (c)-[:MENTIONS]->(p:Person) WHERE NOT toLower(p.name) CONTAINS 'macron' RETURN DISTINCT p.name AS acompanhate_name



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score MATCH (s:Speech)-[:HAS_CHUNK]->(c) MATCH (c)-[:MENTIONS]->(p:Person) WHERE NOT toLower(p.name) CONTAINS 'macron' RETURN DISTINCT p.name AS acompanhate_name"


[{'acompanhate_name': '-'},
 {'acompanhate_name': "Chef de l'État"},
 {'acompanhate_name': 'Jean-Yves LE DRIAN'},
 {'acompanhate_name': 'Sébastien LECORNU'},
 {'acompanhate_name': 'M le Président'},
 {'acompanhate_name': 'Président'},
 {'acompanhate_name': 'mon rôle'},
 {'acompanhate_name': 'Édouard PHILIPPE'},
 {'acompanhate_name': 'président de la Nouvelle-Calédonie'},
 {'acompanhate_name': 'mesdames'},
 {'acompanhate_name': 'Messieurs les Ministres'},
 {'acompanhate_name': 'chers amis'},
 {'acompanhate_name': 'Messieurs les élus'},
 {'acompanhate_name': 'Madame'},
 {'acompanhate_name': 'Emmanuel'},
 {'acompanhate_name': 'le Premier ministre'},
 {'acompanhate_name': 'Président de la République'},
 {'acompanhate_name': 'Monsieur le Premier ministre'},
 {'acompanhate_name': 'le président DUDA'},
 {'acompanhate_name': 'Mateusz'},
 {'acompanhate_name': 'François MITTERRAND'},
 {'acompanhate_name': 'Monsieur le président'},
 {'acompanhate_name': 'Président de la République française'},
 {

Now the final component is to return an answer easy to read for user.  For this step we use `qwen2.5:7b"` to generate the final output.
The function bellow does this task

In [25]:
def answer_user(question: str, graph_results: list, mdodel_llm : str = "qwen2.5:7b") -> str:
    """Génère une réponse claire en français à partir des données Neo4j."""
    prompt = f"""Tu es un assistant spécialisé dans l'analyse des discours présidentiels.
Réponds de manière directe et naturelle à la question de l'utilisateur à partir du résultat brut obtenu dans la base de données Neo4j.

Question : {question}
Données Neo4j : {graph_results}
"""
    response = ollama.chat(
        model=mdodel_llm,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.2}
    )
    return response["message"]["content"].strip()



In [ ]:
# Test du pipeline complet :
question = "Qui était présent avec le président Emanuel Macron lorsqu'il parle des gilets jaunes?"
results = ask_graph(question, NEO4J_SCHEMA_PROMPT, driver)
results


Cypher query
CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score MATCH (s:Speech)-[:HAS_CHUNK]->(c) MATCH (c)-[:MENTIONS]->(p:Person) WHERE NOT toLower(p.name) CONTAINS 'macron' RETURN DISTINCT p.name AS acompanhate_name



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('chunk_embeddings', 100, $query_embedding) YIELD node AS c, score MATCH (s:Speech)-[:HAS_CHUNK]->(c) MATCH (c)-[:MENTIONS]->(p:Person) WHERE NOT toLower(p.name) CONTAINS 'macron' RETURN DISTINCT p.name AS acompanhate_name"


In [ ]:
final_response = answer_user(question, results)
print(f"Réponse finale :\n{final_response}")